# OMIP Simulation Diagnostics -- Multi-case comparison

Post-processing visualization loosely following Adcroft et al. (2019),
*The GFDL Global Ocean and Sea Ice Model OM4.0*, JAMES.

Define two (or more) cases below; every figure shows them side by side.

In [ ]:
cases = [
    (run_dir = "halfdegree_run", prefix = "halfdegree", label = "Half-degree"),
    (run_dir = "orca_run",       prefix = "orca",       label = "ORCA"),
]

In [ ]:
using CairoMakie
using Statistics
using Dates
using Oceananigans
using Oceananigans.Grids: znodes, φnodes
using Oceananigans.Fields: interpolate!
using ConservativeRegridding
using NumericalEarth
using NumericalEarth.DataWrangling: Metadatum
using NumericalEarth.DataWrangling.WOA: WOAAnnual

In [ ]:
# ── Helpers ──────────────────────────────────────────────────

function find_first_file(run_dir, prefix, group)
    tag = "$(prefix)_$(group)"
    candidates = filter(f -> startswith(f, tag) && endswith(f, ".jld2") &&
                             !contains(f, "checkpoint"), readdir(run_dir))
    isempty(candidates) && error("No $group files for prefix '$prefix' in $run_dir")
    return joinpath(run_dir, first(sort(candidates)))
end

function compute_time_mean(fts)
    Nt  = length(fts.times)
    avg = zeros(size(Array(interior(fts[1]))))
    for n in 1:Nt
        avg .+= Array(interior(fts[n]))
    end
    return avg ./ Nt
end

function compute_monthly_mean(fts, target_months; start_date = DateTime(1958, 1, 1))
    dates = [start_date + Second(round(Int, t)) for t in fts.times]
    idx   = findall(d -> month(d) in target_months, dates)
    isempty(idx) && return nothing
    avg = zeros(size(Array(interior(fts[1]))))
    for n in idx
        avg .+= Array(interior(fts[n]))
    end
    return avg ./ length(idx)
end

function build_land_mask(grid)
    if grid isa ImmersedBoundaryGrid
        bh = Array(interior(grid.immersed_boundary.bottom_height, :, :, 1))
        return bh .>= 0
    else
        return falses(size(grid, 1), size(grid, 2))
    end
end

function build_ocean_mask_3d(grid)
    Nx, Ny, Nz = size(grid)
    mask = ones(Nx, Ny, Nz)
    if grid isa ImmersedBoundaryGrid
        bh = Array(interior(grid.immersed_boundary.bottom_height, :, :, 1))
        zc = znodes(grid, Center())
        for k in 1:Nz, j in 1:Ny, i in 1:Nx
            zc[k] < bh[i, j] && (mask[i, j, k] = 0.0)
        end
    end
    return mask
end

mask_land!(f, land) = (f[land] .= NaN; f)

function panel!(fig, pos, data;
                title="", colormap=:thermal,
                colorrange=nothing, label="",
                nan_color=:lightgray)
    ax = Axis(fig[pos...]; title)
    kw = isnothing(colorrange) ? (;) : (; colorrange)
    hm = heatmap!(ax, data; colormap, nan_color, kw...)
    Colorbar(fig[pos[1], pos[2]+1], hm; label)
    return ax
end

case_colors = [:firebrick, :royalblue, :seagreen, :darkorange]

## Load surface diagnostics for all cases

In [ ]:
function load_surface_case(run_dir, prefix)
    surface_file = find_first_file(run_dir, prefix, "surface")
    @info "  surface: $surface_file"

    tos     = FieldTimeSeries(surface_file, "tos";    backend = OnDisk())
    sos     = FieldTimeSeries(surface_file, "sos";    backend = OnDisk())
    zos     = FieldTimeSeries(surface_file, "zos";    backend = OnDisk())
    mld_fts = FieldTimeSeries(surface_file, "mlotst"; backend = OnDisk())
    hfds    = FieldTimeSeries(surface_file, "hfds";   backend = OnDisk())
    wfo     = FieldTimeSeries(surface_file, "wfo";    backend = OnDisk())
    sic     = FieldTimeSeries(surface_file, "siconc"; backend = OnDisk())
    zossq   = FieldTimeSeries(surface_file, "zossq";  backend = OnDisk())

    grid = tos.grid
    Nx, Ny, Nz = size(grid)
    land = build_land_mask(grid)

    SST = dropdims(compute_time_mean(tos);  dims=3)
    SSS = dropdims(compute_time_mean(sos);  dims=3)
    SSH = dropdims(compute_time_mean(zos);  dims=3)
    HF  = dropdims(compute_time_mean(hfds); dims=3)
    FW  = dropdims(compute_time_mean(wfo);  dims=3)
    SIC_mean = dropdims(compute_time_mean(sic); dims=3)

    # SSH variance
    SSH_sq = dropdims(compute_time_mean(zossq); dims=3)
    SSH_var = SSH_sq .- SSH .^ 2

    # MLD min / max over annual cycle
    MLD_monthly = [compute_monthly_mean(mld_fts, [m]) for m in 1:12]
    avail = findall(!isnothing, MLD_monthly)
    MLD_stack = cat([dropdims(MLD_monthly[m]; dims=3) for m in avail]...; dims=3)
    MLD_min = dropdims(minimum(MLD_stack; dims=3); dims=3)
    MLD_max = dropdims(maximum(MLD_stack; dims=3); dims=3)

    # Sea-ice March / September
    SIC_mar = compute_monthly_mean(sic, [3])
    SIC_sep = compute_monthly_mean(sic, [9])
    SIC_mar = isnothing(SIC_mar) ? nothing : dropdims(SIC_mar; dims=3)
    SIC_sep = isnothing(SIC_sep) ? nothing : dropdims(SIC_sep; dims=3)

    # WOA comparison
    T_woa = Field(Metadatum(:temperature; dataset = WOAAnnual()), CPU())
    S_woa = Field(Metadatum(:salinity;    dataset = WOAAnnual()), CPU())
    T_interp = CenterField(grid); interpolate!(T_interp, T_woa)
    S_interp = CenterField(grid); interpolate!(S_interp, S_woa)
    T_woa_on_grid = Array(interior(T_interp))
    S_woa_on_grid = Array(interior(S_interp))
    δSST = SST .- T_woa_on_grid[:, :, Nz]
    δSSS = SSS .- S_woa_on_grid[:, :, Nz]

    # Apply land mask
    for f in (SST, SSS, SSH, HF, FW, SIC_mean, SSH_var, MLD_min, MLD_max, δSST, δSSS)
        mask_land!(f, land)
    end
    !isnothing(SIC_mar) && mask_land!(SIC_mar, land)
    !isnothing(SIC_sep) && mask_land!(SIC_sep, land)

    return (; grid, Nx, Ny, Nz, land, surface_file,
              SST, SSS, SSH, HF, FW, SIC_mean, SSH_var,
              MLD_min, MLD_max, SIC_mar, SIC_sep,
              δSST, δSSS, T_woa_on_grid, S_woa_on_grid)
end

D = Dict{String, Any}()
labels = [c.label for c in cases]
for c in cases
    @info "Loading surface: $(c.label)..."
    D[c.label] = load_surface_case(c.run_dir, c.prefix)
end
@info "Done."

## Figure 1 -- SST bias vs WOA

In [ ]:
fig = Figure(size = (800 * length(labels), 500), fontsize = 14)
for (i, lab) in enumerate(labels)
    panel!(fig, [1, 2i-1], D[lab].δSST;
           title = "$lab: SST - WOA", colormap = :balance,
           colorrange = (-5, 5), label = "deg C")
end
fig

## Figure 2 -- SSS bias vs WOA

In [ ]:
fig = Figure(size = (800 * length(labels), 500), fontsize = 14)
for (i, lab) in enumerate(labels)
    panel!(fig, [1, 2i-1], D[lab].δSSS;
           title = "$lab: SSS - WOA", colormap = :balance,
           colorrange = (-3, 3), label = "PSU")
end
fig

## Figure 3 -- SSH

In [ ]:
fig = Figure(size = (800 * length(labels), 500), fontsize = 14)
for (i, lab) in enumerate(labels)
    panel!(fig, [1, 2i-1], D[lab].SSH;
           title = "$lab: Time-mean SSH", colormap = :balance,
           colorrange = (-2, 2), label = "m")
end
fig

## Figure 4 -- Min / Max monthly-mean MLD (cf. OM4 Figs. 3-4)

In [ ]:
fig = Figure(size = (800 * length(labels), 900), fontsize = 14)
for (i, lab) in enumerate(labels)
    panel!(fig, [1, 2i-1], D[lab].MLD_min;
           title = "$lab: Min MLD (summer)",
           colormap = Reverse(:deep), colorrange = (0, 150), label = "m")
    panel!(fig, [2, 2i-1], D[lab].MLD_max;
           title = "$lab: Max MLD (winter)",
           colormap = Reverse(:deep), colorrange = (10, 3000), label = "m")
end
fig

## Figure 5 -- Sea-ice concentration March / September

In [ ]:
fig = Figure(size = (800 * length(labels), 900), fontsize = 14)
for (i, lab) in enumerate(labels)
    d = D[lab]
    if !isnothing(d.SIC_mar)
        panel!(fig, [1, 2i-1], d.SIC_mar;
               title = "$lab: Sea-ice conc. March",
               colormap = :ice, colorrange = (0, 1), label = "fraction")
    end
    if !isnothing(d.SIC_sep)
        panel!(fig, [2, 2i-1], d.SIC_sep;
               title = "$lab: Sea-ice conc. September",
               colormap = :ice, colorrange = (0, 1), label = "fraction")
    end
end
fig

## Figure 6 -- Surface fluxes

In [ ]:
fig = Figure(size = (800 * length(labels), 900), fontsize = 14)
for (i, lab) in enumerate(labels)
    panel!(fig, [1, 2i-1], D[lab].HF;
           title = "$lab: Net heat flux", colormap = :balance,
           colorrange = (-200, 200), label = "W/m^2")
    panel!(fig, [2, 2i-1], D[lab].FW;
           title = "$lab: Net freshwater flux", colormap = :balance,
           colorrange = (-1e-4, 1e-4), label = "kg/m^2/s")
end
fig

## Figure 7 -- SSH variance

In [ ]:
fig = Figure(size = (800 * length(labels), 500), fontsize = 14)
for (i, lab) in enumerate(labels)
    panel!(fig, [1, 2i-1], D[lab].SSH_var;
           title = "$lab: SSH variance", colormap = :magma,
           colorrange = (0, 0.05), label = "m\u00b2")
end
fig

## Load time series and 3-D fields for all cases

In [ ]:
function load_timeseries_case(run_dir, prefix, grid)
    # Global-mean drift
    avg_file = find_first_file(run_dir, prefix, "averages")
    tosga_fts = FieldTimeSeries(avg_file, "tosga"; backend = OnDisk())
    soga_fts  = FieldTimeSeries(avg_file, "soga";  backend = OnDisk())
    tosga = [Array(interior(tosga_fts[n]))[1] for n in 1:length(tosga_fts.times)]
    soga  = [Array(interior(soga_fts[n]))[1]  for n in 1:length(soga_fts.times)]
    t_avg = tosga_fts.times ./ (365.25 * 24 * 3600)

    # Horizontal-mean profiles
    to_h_fts = FieldTimeSeries(avg_file, "to_h"; backend = OnDisk())
    so_h_fts = FieldTimeSeries(avg_file, "so_h"; backend = OnDisk())
    T_prof = vec(compute_time_mean(to_h_fts))
    S_prof = vec(compute_time_mean(so_h_fts))
    z = collect(znodes(grid, Center()))

    # Global-mean TKE
    fields_file = find_first_file(run_dir, prefix, "fields")
    tke_fts = FieldTimeSeries(fields_file, "tke"; backend = OnDisk())
    ocean_mask = build_ocean_mask_3d(grid)
    N_ocean = sum(ocean_mask)
    tke_global = [sum(Array(interior(tke_fts[n])) .* ocean_mask) / N_ocean
                  for n in 1:length(tke_fts.times)]
    t_tke = tke_fts.times ./ (365.25 * 24 * 3600)

    return (; tosga, soga, t_avg, T_prof, S_prof, z,
              tke_global, t_tke, ocean_mask, fields_file)
end

TS = Dict{String, Any}()
for c in cases
    @info "Loading time series: $(c.label)..."
    TS[c.label] = load_timeseries_case(c.run_dir, c.prefix, D[c.label].grid)
end
@info "Done."

## Figure 8 -- Global-mean TKE

In [ ]:
fig = Figure(size = (900, 450), fontsize = 14)
ax = Axis(fig[1, 1]; xlabel = "Time (years)", ylabel = "TKE (m\u00b2/s\u00b2)",
          title = "Global-mean turbulent kinetic energy")
for (i, lab) in enumerate(labels)
    lines!(ax, TS[lab].t_tke, TS[lab].tke_global;
           color = case_colors[i], label = lab)
end
axislegend(ax; position = :rb)
fig

## Figure 9 -- Global-mean T and S drift

In [ ]:
fig = Figure(size = (1200, 450), fontsize = 14)
ax = Axis(fig[1, 1]; xlabel = "Time (years)", ylabel = "\u0394T (deg C)",
          title = "Global-mean temperature drift")
for (i, lab) in enumerate(labels)
    d = TS[lab]
    lines!(ax, d.t_avg, d.tosga .- d.tosga[1]; color = case_colors[i], label = lab)
end
axislegend(ax; position = :lb)

ax = Axis(fig[1, 2]; xlabel = "Time (years)", ylabel = "\u0394S (PSU)",
          title = "Global-mean salinity drift")
for (i, lab) in enumerate(labels)
    d = TS[lab]
    lines!(ax, d.t_avg, d.soga .- d.soga[1]; color = case_colors[i], label = lab)
end
axislegend(ax; position = :lb)
fig

## Figure 10 -- Horizontal-mean T and S profiles

In [ ]:
fig = Figure(size = (1000, 600), fontsize = 14)
ax = Axis(fig[1, 1]; xlabel = "Temperature (deg C)", ylabel = "Depth (m)",
          title = "Horizontal-mean temperature")
for (i, lab) in enumerate(labels)
    lines!(ax, TS[lab].T_prof, TS[lab].z; color = case_colors[i], label = lab)
end
ylims!(ax, (-5500, 0)); axislegend(ax; position = :rb)

ax = Axis(fig[1, 2]; xlabel = "Salinity (PSU)", ylabel = "Depth (m)",
          title = "Horizontal-mean salinity")
for (i, lab) in enumerate(labels)
    lines!(ax, TS[lab].S_prof, TS[lab].z; color = case_colors[i], label = lab)
end
ylims!(ax, (-5500, 0)); axislegend(ax; position = :rb)
fig

---
## Zonal-mean sections

Regrid 3-D time-mean fields to a regular 1-degree lat-lon grid via
`ConservativeRegridding`, then average over longitude.
A per-case regridder is built (this is expensive).

In [ ]:
Nlon, Nlat = 360, 180
latlon_grid = LatitudeLongitudeGrid(CPU();
    size = (Nlon, Nlat, 1),
    longitude = (0, 360),
    latitude  = (-90, 90),
    z = (0, 1))

dst_f = Field{Center, Center, Nothing}(latlon_grid)

"""Regrid a 3-D field level-by-level and compute the
ocean-area-weighted zonal mean using a carried ocean mask."""
function compute_zonal_mean(data_3d, ocean_mask_3d, regridder, Nlon, Nlat)
    Nz = size(data_3d, 3)
    zonal    = fill(NaN, Nlat, Nz)
    dst_data = zeros(Nlon * Nlat)
    dst_mask = zeros(Nlon * Nlat)
    areas    = regridder.dst_areas
    for k in 1:Nz
        ConservativeRegridding.regrid!(dst_data, regridder,
            vec(data_3d[:, :, k] .* ocean_mask_3d[:, :, k]))
        ConservativeRegridding.regrid!(dst_mask, regridder,
            vec(ocean_mask_3d[:, :, k]))
        data_sum = reshape(dst_data .* areas, Nlon, Nlat)
        mask_sum = reshape(dst_mask .* areas, Nlon, Nlat)
        for j in 1:Nlat
            m = sum(@view mask_sum[:, j])
            m > 0 && (zonal[j, k] = sum(@view data_sum[:, j]) / m)
        end
    end
    return zonal
end

In [ ]:
ZM = Dict{String, Any}()

for c in cases
    lab  = c.label
    grid = D[lab].grid
    ocean_mask = TS[lab].ocean_mask

    # Build per-case regridder
    @info "Building regridder for $lab (may take a few minutes)..."
    src_f = Field{Center, Center, Nothing}(grid)
    regridder = ConservativeRegridding.Regridder(dst_f, src_f; progress = true)

    # Load 3-D fields
    @info "Loading 3-D fields for $lab..."
    fields_file = TS[lab].fields_file
    to_fts = FieldTimeSeries(fields_file, "to"; backend = OnDisk())
    so_fts = FieldTimeSeries(fields_file, "so"; backend = OnDisk())
    bo_fts = FieldTimeSeries(fields_file, "bo"; backend = OnDisk())

    T_mean = compute_time_mean(to_fts)
    S_mean = compute_time_mean(so_fts)
    b_mean = compute_time_mean(bo_fts)
    b_init = Array(interior(bo_fts[1]))

    @info "Computing zonal means for $lab..."
    T_z  = compute_zonal_mean(T_mean, ocean_mask, regridder, Nlon, Nlat)
    S_z  = compute_zonal_mean(S_mean, ocean_mask, regridder, Nlon, Nlat)
    b_z  = compute_zonal_mean(b_mean, ocean_mask, regridder, Nlon, Nlat)

    Tw_z = compute_zonal_mean(D[lab].T_woa_on_grid, ocean_mask, regridder, Nlon, Nlat)
    Sw_z = compute_zonal_mean(D[lab].S_woa_on_grid, ocean_mask, regridder, Nlon, Nlat)
    bi_z = compute_zonal_mean(b_init,               ocean_mask, regridder, Nlon, Nlat)

    z = collect(znodes(grid, Center()))

    ZM[lab] = (; T_z, S_z, b_z,
                δT_z = T_z .- Tw_z,
                δS_z = S_z .- Sw_z,
                δb_z = b_z .- bi_z, z)
end

φ = collect(φnodes(latlon_grid, Center()))
@info "Zonal means done."

## Figure 11 -- Zonal-mean T, S, b

In [ ]:
fig = Figure(size = (600 * length(labels), 900), fontsize = 14)
for (i, lab) in enumerate(labels)
    zm = ZM[lab]
    ax = Axis(fig[1, 2i-1]; xlabel="Latitude", ylabel="Depth (m)",
              title="$lab: Zonal T")
    hm = heatmap!(ax, φ, zm.z, zm.T_z; colormap=:thermal,
                  colorrange=(-2,30), nan_color=:lightgray)
    Colorbar(fig[1, 2i], hm; label="deg C")
    ylims!(ax, (-5500, 0))

    ax = Axis(fig[2, 2i-1]; xlabel="Latitude", ylabel="Depth (m)",
              title="$lab: Zonal S")
    hm = heatmap!(ax, φ, zm.z, zm.S_z; colormap=:haline,
                  colorrange=(33,37), nan_color=:lightgray)
    Colorbar(fig[2, 2i], hm; label="PSU")
    ylims!(ax, (-5500, 0))

    ax = Axis(fig[3, 2i-1]; xlabel="Latitude", ylabel="Depth (m)",
              title="$lab: Zonal b")
    hm = heatmap!(ax, φ, zm.z, zm.b_z; colormap=:balance,
                  nan_color=:lightgray)
    Colorbar(fig[3, 2i], hm; label="m/s\u00b2")
    ylims!(ax, (-5500, 0))
end
fig

## Figure 12 -- Zonal-mean drift from initial conditions

In [ ]:
fig = Figure(size = (600 * length(labels), 900), fontsize = 14)
for (i, lab) in enumerate(labels)
    zm = ZM[lab]
    ax = Axis(fig[1, 2i-1]; xlabel="Latitude", ylabel="Depth (m)",
              title="$lab: Zonal T - WOA")
    hm = heatmap!(ax, φ, zm.z, zm.δT_z; colormap=:balance,
                  colorrange=(-5,5), nan_color=:lightgray)
    Colorbar(fig[1, 2i], hm; label="deg C")
    ylims!(ax, (-5500, 0))

    ax = Axis(fig[2, 2i-1]; xlabel="Latitude", ylabel="Depth (m)",
              title="$lab: Zonal S - WOA")
    hm = heatmap!(ax, φ, zm.z, zm.δS_z; colormap=:balance,
                  colorrange=(-1,1), nan_color=:lightgray)
    Colorbar(fig[2, 2i], hm; label="PSU")
    ylims!(ax, (-5500, 0))

    ax = Axis(fig[3, 2i-1]; xlabel="Latitude", ylabel="Depth (m)",
              title="$lab: Zonal b - b(t=0)")
    hm = heatmap!(ax, φ, zm.z, zm.δb_z; colormap=:balance,
                  nan_color=:lightgray)
    Colorbar(fig[3, 2i], hm; label="m/s\u00b2")
    ylims!(ax, (-5500, 0))
end
fig